# Data contracts and validation

Read `app/goals.py`, `app/models.py` and `app/errors.py` alongside this lesson. Predict the output before running each cell. These examples use the real domain classes without importing the HTTP application or loading secrets.

Learning goals: distinguish configuration from observations and state; understand field and cross-field validation; explain why schema validity is not truth.


In [1]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'app' / 'planner.py').exists()), None)
if root is None:
    raise RuntimeError('Open this notebook from the repository or docs/learning directory.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print('Repository imports ready. No provider calls or application server will be started.')


Repository imports ready. No provider calls or application server will be started.


## Configuration is a contract

The default configuration includes a catalogue of goals. `kind` controls whether a typed choice or number is required. `condition` controls applicability; a disabled or excluded goal is different from an unanswered goal.


In [2]:
from app.models import AgentConfig
from pydantic import ValidationError
cfg = AgentConfig()
print('Agent:', cfg.name, '| Goals:', len(cfg.goals))
for g in cfg.goals:
    condition = f'{g.condition.goal_key} == {g.condition.equals}' if g.condition else 'always'
    print(g.key, g.kind, 'required' if g.required else 'optional', condition)
assert len(cfg.goals) == 15


Agent: Mira | Goals: 15
startup_name text required always
sector choice required always
problem text required always
target_customer text required always
team text required always
product text required always
stage choice required always
traction text required always
competitors text required always
fundraising choice required always
funding_amount number required fundraising == Ja
funding_round choice required fundraising == Ja
use_of_funds text required fundraising == Ja
funding_timeline text optional fundraising == Ja
next_step text required always


## Individual fields and relationships are both checked

Out-of-range speed is a field error. Duplicate keys and forward conditions require cross-field validation. A plain type hint would not provide any of these runtime checks.


In [3]:
def show_validation(label, data):
    try:
        AgentConfig.model_validate(data)
    except ValidationError as exc:
        print(label, 'rejected:', exc.errors()[0]['msg'])
    else:
        raise AssertionError('Expected invalid configuration')

show_validation('Invalid speed', {**cfg.model_dump(), 'speed': 99})
duplicate = cfg.model_dump()
duplicate['goals'][1]['key'] = duplicate['goals'][0]['key']
show_validation('Duplicate goal key', duplicate)
forward = cfg.model_dump()
forward['goals'][0]['condition'] = {'goal_key':'fundraising', 'equals':'Ja'}
show_validation('Forward dependency', forward)
show_validation('Unexpected secret field', {**cfg.model_dump(), 'api_key':'fictional-not-a-real-key'})


Invalid speed rejected: Input should be less than or equal to 1.2
Duplicate goal key rejected: Value error, Unique goal keys and at least one enabled required goal are needed
Forward dependency rejected: Value error, Conditions must refer to an earlier choice goal and one of its options
Unexpected secret field rejected: Extra inputs are not permitted


## Configuration, answer and state

`GoalAnswer` proposes one interpretation. `GoalState` is accumulated progress. Creating a structurally valid answer does not prove it is supported by the supplied quote. The next lesson shows where that evidence check happens.


In [4]:
from app.goals import GoalAnswer, GoalState
answer = GoalAnswer(goal_key='startup_name', status='answered', value='Nordlicht', quote='Nordlicht')
state = GoalState()
print('Proposed answer:', answer.model_dump())
print('Unchanged state:', state.model_dump())
assert state.status == 'open'
assert state.value is None
unproven = answer.model_copy(update={'value':'A completely different name'})
print('A structural model is not a semantic proof:', unproven.value, '| quote:', unproven.quote)


Proposed answer: {'goal_key': 'startup_name', 'status': 'answered', 'value': 'Nordlicht', 'quote': 'Nordlicht', 'number_value': None, 'choice_value': None}
Unchanged state: {'status': 'open', 'value': None, 'quote': None, 'evidence_turn': None, 'number_value': None, 'choice_value': None, 'asked': False, 'followups_used': 0, 'exhausted': False}
A structural model is not a semantic proof: A completely different name | quote: Nordlicht


## Snapshot serialization

The API persists a serialized configuration with the interview. Reconstructing it yields separate objects. Do not assume a shallow Python assignment creates a historical snapshot.


In [5]:
saved_json = cfg.model_dump_json()
restored = AgentConfig.model_validate_json(saved_json)
cfg.name = 'New administrator draft'
assert restored.name == 'Mira'
assert cfg.name != restored.name
print('Draft:', cfg.name, '| Restored snapshot:', restored.name)


Draft: New administrator draft | Restored snapshot: Mira


## Exercises and interview answers

1. Change speed to 1.1. Why should that pass?
2. Move `fundraising` after `funding_amount` and validate again. Why does order matter?
3. Can `GoalAnswer(value='one million', number_value=None)` prove a numeric answer is complete? No; the planner knows the target goal's type and enforces that contract.
4. Explain `None` versus `0` for a funding amount. Unknown is not zero.
5. Explain why extra fields are forbidden but some compatible values may still be coerced by Pydantic. `StrictModel` is a project class name, not global strict-type mode.
